# Run training on the cluster (validate Slurm → launch Gemma)

## Objective
With a cluster up (from `01-provision-cluster.ipynb`), connect to the login node,
prove the scheduler works with a smoke-test job, then launch the Gemma recipe on
whatever hardware won the fallback (PDF Steps 3.5–4.6).

Docs: [Run prebuilt workloads](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/training/training-clusters/run-prebuilt-workloads) ·
[Gemma distributed tuning](https://ai.google.dev/gemma/docs/core/distributed_tuning)

## 1. Recover cluster state + parameters

In [1]:
import yaml, pathlib, os, subprocess

cfg = yaml.safe_load(open("../configs/cluster_config.yaml"))
PROJECT = cfg["project"]["id"]; REGION = cfg["project"]["region"]; ZONE = cfg["project"]["zone"]
BUCKET = cfg["bucket"]["name"] or f"{PROJECT}-vtc-temp"

state = {}
sp = pathlib.Path(".vtc_cluster_state")
if sp.exists():
    for line in sp.read_text().splitlines():
        if "=" in line:
            k, v = line.split("=", 1); state[k] = v
CLUSTER_ID = state.get("CLUSTER_ID", "vtcgemma")
PROFILE    = state.get("PROFILE", "cpu")  # also the Slurm partition name
ACCEL = next((p["accelerator"] for p in cfg["fallback_chain"] if p["name"] == PROFILE), "cpu")

# Avoid the two notebook SSH hangs without interactive prompts:
#  1. pre-create the gcloud SSH key with an EMPTY passphrase (no "Enter passphrase").
#  2. disable host-key checking via gcloud's NATIVE flag --strict-host-key-checking=no,
#     which works for BOTH `gcloud compute ssh` AND `gcloud compute scp`
#     (scp does NOT accept the `-- -o ...` ssh passthrough — that errors).
_key = os.path.expanduser("~/.ssh/google_compute_engine")
if not os.path.exists(_key):
    subprocess.run(["ssh-keygen", "-t", "rsa", "-N", "", "-f", _key, "-q"], check=True)
    print("created SSH key:", _key)
SSH_OPTS = "--strict-host-key-checking=no"

print(f"cluster={CLUSTER_ID} winning_profile={PROFILE} accelerator={ACCEL}")

cluster=vtcgemma winning_profile=cpu accelerator=cpu


## 2. Find the login node + SSH in

Find the login node IP under **Vertex AI > Training > Model Development Clusters**,
or SSH straight to the named VM. The login VM is `<cluster-id>-login-001`.

> **SSH needs OS Login — least-privilege role is `roles/compute.osLogin`.**

In [2]:
LOGIN = f"{CLUSTER_ID}-login-001"
print(f"Login node: {LOGIN}\n")

# Validate the scheduler over SSH — you should see the partition named PROFILE.
# --quiet + SSH_OPTS keep it non-interactive (no passphrase/host-key prompts).
# First connect may take ~30-60s while the key propagates to the VM.
!gcloud compute ssh {LOGIN} --project={PROJECT} --zone={ZONE} --quiet \
    --command='echo "== sinfo =="; sinfo; echo "== squeue =="; squeue' {SSH_OPTS}

Login node: vtcgemma-login-001

== sinfo ==
PARTITION AVAIL  TIMELIMIT  NODES  STATE NODELIST
cpu*         up   infinite      2   idle vtcgemma-cpu-[0-1]
== squeue ==
             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


## 3. Smoke-test job

Write the SBATCH script with `%%writefile`-style write, then **the cell copies it to
the login node and submits it over SSH**. It places a one-task job on a worker and
writes back to the shared `/home` — the end-to-end proof the cluster works.
`--partition` is set to the winning profile automatically.

In [3]:
smoke = f'''#!/bin/bash
#SBATCH --job-name=smoke_test
#SBATCH --partition={PROFILE}
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --output=smoke_%j.out

echo "Hello from $(hostname)"
sleep 30
echo "Job finished"
'''
open("smoke_test.sh", "w").write(smoke)
LOGIN = f"{CLUSTER_ID}-login-001"
print(smoke)

# Copy + submit on the login node, then wait and show the output (non-interactive SSH).
!gcloud compute scp smoke_test.sh {LOGIN}:~ --project={PROJECT} --zone={ZONE} --quiet {SSH_OPTS}
!gcloud compute ssh {LOGIN} --project={PROJECT} --zone={ZONE} --quiet \
    --command='sbatch smoke_test.sh && sleep 35 && cat smoke_*.out' {SSH_OPTS}

#!/bin/bash
#SBATCH --job-name=smoke_test
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --output=smoke_%j.out

echo "Hello from $(hostname)"
sleep 30
echo "Job finished"

smoke_test.sh                                 100%  202    16.7KB/s   00:00    
Submitted batch job 1
Hello from vtcgemma-cpu-0
Job finished


## 4. Stage the Gemma training code

GPU clusters train with NeMo-Run; TPU clusters with JAX/MaxText. Stage the recipe
tree into the code-transfer bucket, then pull it onto the login node. (Source the
recipe tree from your model-garden / NeMo checkout — see README "Stage training
code".)

In [4]:
print("Stage recipes to the bucket (run where you have the recipe tree):")
if ACCEL == "tpu":
    print(f"  gcloud storage cp -r <path-to>/maxtext gs://{BUCKET}/maxtext/")
else:
    print(f"  gcloud storage cp -r <path-to>/nemo gs://{BUCKET}/nemo/")
print("\nThe launcher pulls from the bucket onto the login node automatically.")

Stage recipes to the bucket (run where you have the recipe tree):
  gcloud storage cp -r <path-to>/nemo gs://sandbox-401718-vtc-temp/nemo/

The launcher pulls from the bucket onto the login node automatically.


## 5. Launch Gemma

This is the **on-node step both paths share**. The Terraform demo (Path A) stops at
*cluster-ready*, so the recipe is launched the same way there and here: push the
shared `scripts/jobs/launch_gemma.sh` to the login node and run it. It picks NeMo
(GPU) or MaxText (TPU) from the winning profile and runs the recipe named in the
config. For a `cpu` fallback there's no training fabric — it no-ops back to the
smoke test.

Set **`MOCK_DATA = True`** below for the *SparkPi of training*: the real Gemma code on
**random tokens** (NeMo `MockDataModule`) — no dataset/bucket needed, GPU only. It proves
the stack runs end-to-end but learns nothing (throwaway weights).

In [ ]:
gpu = cfg["workload"]["gpu"]; tpu = cfg["workload"]["tpu"]
recipe = tpu["recipe"] if ACCEL == "tpu" else gpu["recipe"]
DATA_DIR = cfg["workload"]["data_dir"]
LOGIN = f"{CLUSTER_ID}-login-001"

MOCK_DATA = False  # @param {type:"boolean"}  — random tokens, no dataset (GPU smoke test)

if MOCK_DATA:
    # "SparkPi" of training: real Gemma code on random tokens (NeMo MockDataModule).
    # No bucket/dataset/recipe on GPU. Useless weights — just proves the stack runs.
    launch = f"bash launch_gemma.sh --accelerator {ACCEL} --mock-data --nodes 1 --gpus-per-node 8"
else:
    launch = (f"bash launch_gemma.sh --accelerator {ACCEL} --bucket gs://{BUCKET} "
              f"--recipe {recipe} --slurm-type {gpu['slurm_type']} --data-dir {DATA_DIR}")
print("Launching via the shared scripts/jobs/launch_gemma.sh on", LOGIN, "\n ", launch, "\n")

# Push the shared launcher (+ the mock recipe) to the login node and run it there.
# Requires the cluster up. On a cpu cluster this no-ops (no training fabric).
!gcloud compute scp ../scripts/jobs/launch_gemma.sh ../scripts/jobs/gemma_mock_pretrain.py {LOGIN}:~ --project={PROJECT} --zone={ZONE} --quiet {SSH_OPTS}
!gcloud compute ssh {LOGIN} --project={PROJECT} --zone={ZONE} --quiet \
    --command='{launch}' {SSH_OPTS}

print("\nMonitor on the login node: squeue ; sinfo ; sacct   (logs in ~/my_training_run)")

## Next
- **`03-teardown.ipynb`** — tear it all down when you're done (accelerators bill
  while the cluster exists).